# Condensed Nearest Neighbour + 모델 2개 성능 비교

이 notebook은 `docs/기법별_성능평가.md`의 평가 계획에 맞춰 `Undersampling CNN + 모델 2개` 실험을 준비하는 템플릿입니다.


## 목적

CondensedNearestNeighbour(CNN)로 다수 클래스 중 분류 경계에 필요한 샘플만 남겼을 때 모델 1 성능 변화를 확인한다.

여기서 CNN은 합성곱 신경망이 아니라 imbalanced-learn의 CondensedNearestNeighbour이다. 전체 데이터에서는 오래 걸릴 수 있다.


## 필요한 입력 파일

- `data/processed/original_train_train_preprocessed.csv`
- `data/processed/original_train_val_preprocessed.csv`
- `data/processed/original_train_test_preprocessed.csv`
- `artifacts/preprocessors/original_train_preprocess_config.json`

## 실행 절차

1. 입력 파일과 feature 목록을 확인한다.
2. 모델 1 학습용 train split에만 imbalanced-learn sampling을 적용한다.
3. sampling된 train CSV를 저장한다.
4. 모델 1은 sampling된 train CSV로 학습한다.
5. 모델 2는 sampling을 적용하지 않고, 기존 스크립트가 `위험도 > 0` 데이터만 필터링하여 학습한다.
6. 결과 지표를 계산하고 `docs/기법별_성능평가.md` 평가표에 옮겨 적는다.

## 모델 2 주의사항

모델 2 회귀모델은 항상 `위험도 > 0` 데이터만 사용한다. Sampling 기법은 모델 1의 `사고발생` 분류 문제에만 적용한다.


In [4]:
from __future__ import annotations

import json
import math
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, mean_absolute_error, mean_squared_error

from imblearn.under_sampling import CondensedNearestNeighbour

PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'src').exists()
)

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
TRAIN_PATH = DATA_DIR / 'original_train_train_preprocessed.csv'
VAL_PATH = DATA_DIR / 'original_train_val_preprocessed.csv'
TEST_PATH = DATA_DIR / 'original_train_test_preprocessed.csv'
PREPROCESS_CONFIG_PATH = PROJECT_ROOT / 'artifacts' / 'preprocessors' / 'original_train_preprocess_config.json'

EXPERIMENT_NAME = 'Undersampling CNN + 모델 2개'
EXPERIMENT_SLUG = 'condensed_nearest_neighbour_two_stage'
RANDOM_STATE = 42
ROW_LIMIT = 50000  # None이면 전체 train 데이터를 사용한다.
RUN_SAMPLING = True  # 실제 sampling CSV를 만들려면 True로 변경한다.
RUN_TRAINING = True  # 실제 모델 학습까지 실행하려면 True로 변경한다.

OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'experiments' / EXPERIMENT_SLUG
SAMPLED_INPUT_DIR = PROJECT_ROOT / 'artifacts' / 'experiments' / 'sampling_inputs'
SAMPLED_TRAIN_PATH = SAMPLED_INPUT_DIR / f'{EXPERIMENT_SLUG}_model1_train_preprocessed.csv'

MODEL1_MODEL_PATH = PROJECT_ROOT / 'artifacts' / 'models' / f'mlp_accident_classifier_{EXPERIMENT_SLUG}.keras'
MODEL1_METRICS_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / f'mlp_accident_classifier_{EXPERIMENT_SLUG}_metrics.json'
MODEL1_HISTORY_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / f'mlp_accident_classifier_{EXPERIMENT_SLUG}_history.csv'
MODEL1_PREDICTIONS_PATH = PROJECT_ROOT / 'artifacts' / 'predictions' / f'mlp_accident_classifier_{EXPERIMENT_SLUG}_test_predictions.csv'

MODEL2_MODEL_PATH = PROJECT_ROOT / 'artifacts' / 'models' / f'mlp_positive_risk_regressor_{EXPERIMENT_SLUG}.keras'
MODEL2_METRICS_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / f'mlp_positive_risk_regressor_{EXPERIMENT_SLUG}_metrics.json'
MODEL2_HISTORY_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / f'mlp_positive_risk_regressor_{EXPERIMENT_SLUG}_history.csv'
MODEL2_PREDICTIONS_PATH = PROJECT_ROOT / 'artifacts' / 'predictions' / f'mlp_positive_risk_regressor_{EXPERIMENT_SLUG}_test_predictions.csv'

EXPERIMENT_RESULT_PATH = OUTPUT_DIR / f'{EXPERIMENT_SLUG}_summary.csv'
DOCS_TABLE_PATH = PROJECT_ROOT / 'docs' / '기법별_성능평가.md'


In [5]:
required_paths = {
    'train': TRAIN_PATH,
    'val': VAL_PATH,
    'test': TEST_PATH,
    'preprocess_config': PREPROCESS_CONFIG_PATH,
    'docs_table': DOCS_TABLE_PATH,
}

path_status = pd.DataFrame(
    [{'name': name, 'path': str(path.relative_to(PROJECT_ROOT)), 'exists': path.exists()} for name, path in required_paths.items()]
)
display(path_status)

missing = path_status.loc[~path_status['exists'], 'path'].tolist()
if missing:
    raise FileNotFoundError(f'필요한 파일이 없습니다: {missing}')

with PREPROCESS_CONFIG_PATH.open('r', encoding='utf-8') as file:
    preprocess_config = json.load(file)

feature_columns = preprocess_config['feature_columns']
print(f'feature 수: {len(feature_columns):,}')


,name,path,exists
0,train,data/processed/original_train_train_preprocess...,True
1,val,data/processed/original_train_val_preprocessed...,True
2,test,data/processed/original_train_test_preprocesse...,True
3,preprocess_config,artifacts/preprocessors/original_train_preproc...,True
4,docs_table,docs/기법별_성능평가.md,True


feature 수: 102


In [6]:
def accident_target(frame: pd.DataFrame) -> pd.Series:
    """위험도 값을 모델 1의 사고발생 이진 라벨로 변환한다."""
    return (frame['위험도'] > 0).astype(int)


train_frame = pd.read_csv(TRAIN_PATH, nrows=ROW_LIMIT)
X_train = train_frame[feature_columns]
y_train = accident_target(train_frame)

print('sampling 실행 전 클래스 분포')
display(y_train.value_counts().rename(index={0: '사고발생=0', 1: '사고발생=1'}).to_frame('count'))

sampler = CondensedNearestNeighbour(random_state=RANDOM_STATE, n_jobs=-1)
print(sampler)

if RUN_SAMPLING:
    X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
    sampled_train = pd.DataFrame(X_resampled, columns=feature_columns)

    # 모델 1 학습 스크립트는 위험도 > 0 여부만 사용하므로, sampling 후 라벨 복원을 위해 위험도 컬럼을 만든다.
    sampled_train['위험도'] = np.where(np.asarray(y_resampled) == 1, 1.0, 0.0)

    SAMPLED_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    sampled_train.to_csv(SAMPLED_TRAIN_PATH, index=False)

    print('sampling 실행 후 클래스 분포')
    display(pd.Series(y_resampled).value_counts().rename(index={0: '사고발생=0', 1: '사고발생=1'}).to_frame('count'))
    print(f'sampled train 저장: {SAMPLED_TRAIN_PATH.relative_to(PROJECT_ROOT)}')
else:
    print('RUN_SAMPLING=False 상태입니다. 실제 sampling CSV 생성은 실행하지 않았습니다.')
    print('실행하려면 위 설정 셀에서 RUN_SAMPLING=True로 변경하십시오.')


sampling 실행 전 클래스 분포


,count
위험도,
사고발생=0,41291
사고발생=1,8709


CondensedNearestNeighbour(n_jobs=-1, random_state=42)


/tmp/ipykernel_36323/2135345501.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sampled_train['위험도'] = np.where(np.asarray(y_resampled) == 1, 1.0, 0.0)


sampling 실행 후 클래스 분포


,count
위험도,
사고발생=1,8709
사고발생=0,5159


sampled train 저장: artifacts/experiments/sampling_inputs/condensed_nearest_neighbour_two_stage_model1_train_preprocessed.csv


In [7]:
model1_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'train' / 'train_accident_classifier.py'),
    '--train', str(SAMPLED_TRAIN_PATH),
    '--val', str(VAL_PATH),
    '--test', str(TEST_PATH),
    '--model-path', str(MODEL1_MODEL_PATH),
    '--metrics-path', str(MODEL1_METRICS_PATH),
    '--history-path', str(MODEL1_HISTORY_PATH),
    '--predictions-path', str(MODEL1_PREDICTIONS_PATH),
    '--epochs', '100',
    '--batch-size', '1024',
    '--device', 'auto',
    '--verbose', '2',
    '--top-k', '1000',
]

print('모델 1 학습 명령:')
print(' '.join(model1_command))

if RUN_TRAINING:
    if not SAMPLED_TRAIN_PATH.exists():
        raise FileNotFoundError('모델 1 학습 전 RUN_SAMPLING=True로 sampling train CSV를 먼저 생성해야 합니다.')
    subprocess.run(model1_command, check=True)
else:
    print('RUN_TRAINING=False 상태입니다. 실제 모델 1 학습은 실행하지 않았습니다.')


모델 1 학습 명령:
/home/huichan/SPARV/SilverWalk/.venv/bin/python /home/huichan/SPARV/SilverWalk/scripts/train/train_accident_classifier.py --train /home/huichan/SPARV/SilverWalk/artifacts/experiments/sampling_inputs/condensed_nearest_neighbour_two_stage_model1_train_preprocessed.csv --val /home/huichan/SPARV/SilverWalk/data/processed/original_train_val_preprocessed.csv --test /home/huichan/SPARV/SilverWalk/data/processed/original_train_test_preprocessed.csv --model-path /home/huichan/SPARV/SilverWalk/artifacts/models/mlp_accident_classifier_condensed_nearest_neighbour_two_stage.keras --metrics-path /home/huichan/SPARV/SilverWalk/artifacts/reports/mlp_accident_classifier_condensed_nearest_neighbour_two_stage_metrics.json --history-path /home/huichan/SPARV/SilverWalk/artifacts/reports/mlp_accident_classifier_condensed_nearest_neighbour_two_stage_history.csv --predictions-path /home/huichan/SPARV/SilverWalk/artifacts/predictions/mlp_accident_classifier_condensed_nearest_neighbour_two_stage_tes

I0000 00:00:1780786864.225063   53835 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780786864.418493   53835 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780786865.626973   53835 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1780786867.075636   53835 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please mak

Epoch 1/100
14/14 - 1s - 84ms/step - accuracy: 0.5699 - loss: 0.7318 - pr_auc: 0.6776 - precision: 0.6661 - recall: 0.6319 - roc_auc: 0.5679 - val_accuracy: 0.4741 - val_loss: 0.7240 - val_pr_auc: 0.3587 - val_precision: 0.2532 - val_recall: 0.8398 - val_roc_auc: 0.6885 - learning_rate: 0.0010
Epoch 2/100
14/14 - 0s - 15ms/step - accuracy: 0.6331 - loss: 0.6507 - pr_auc: 0.7360 - precision: 0.6700 - recall: 0.8192 - roc_auc: 0.6361 - val_accuracy: 0.4800 - val_loss: 0.7176 - val_pr_auc: 0.3828 - val_precision: 0.2578 - val_recall: 0.8557 - val_roc_auc: 0.7094 - learning_rate: 0.0010
Epoch 3/100
14/14 - 0s - 14ms/step - accuracy: 0.6495 - loss: 0.6286 - pr_auc: 0.7582 - precision: 0.6808 - recall: 0.8320 - roc_auc: 0.6603 - val_accuracy: 0.4793 - val_loss: 0.7176 - val_pr_auc: 0.3942 - val_precision: 0.2576 - val_recall: 0.8562 - val_roc_auc: 0.7154 - learning_rate: 0.0010
Epoch 4/100
14/14 - 0s - 14ms/step - accuracy: 0.6542 - loss: 0.6220 - pr_auc: 0.7669 - precision: 0.6876 - recall:

In [8]:
model2_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'train' / 'train_positive_risk_regressor.py'),
    '--train', str(TRAIN_PATH),
    '--val', str(VAL_PATH),
    '--test', str(TEST_PATH),
    '--model-path', str(MODEL2_MODEL_PATH),
    '--metrics-path', str(MODEL2_METRICS_PATH),
    '--history-path', str(MODEL2_HISTORY_PATH),
    '--predictions-path', str(MODEL2_PREDICTIONS_PATH),
    '--epochs', '100',
    '--batch-size', '1024',
    '--device', 'auto',
    '--verbose', '2',
]

print('모델 2 학습 명령:')
print(' '.join(model2_command))
print('주의: 모델 2 학습 스크립트는 내부에서 위험도 > 0 데이터만 필터링합니다.')

if RUN_TRAINING:
    subprocess.run(model2_command, check=True)
else:
    print('RUN_TRAINING=False 상태입니다. 실제 모델 2 학습은 실행하지 않았습니다.')


모델 2 학습 명령:
/home/huichan/SPARV/SilverWalk/.venv/bin/python /home/huichan/SPARV/SilverWalk/scripts/train/train_positive_risk_regressor.py --train /home/huichan/SPARV/SilverWalk/data/processed/original_train_train_preprocessed.csv --val /home/huichan/SPARV/SilverWalk/data/processed/original_train_val_preprocessed.csv --test /home/huichan/SPARV/SilverWalk/data/processed/original_train_test_preprocessed.csv --model-path /home/huichan/SPARV/SilverWalk/artifacts/models/mlp_positive_risk_regressor_condensed_nearest_neighbour_two_stage.keras --metrics-path /home/huichan/SPARV/SilverWalk/artifacts/reports/mlp_positive_risk_regressor_condensed_nearest_neighbour_two_stage_metrics.json --history-path /home/huichan/SPARV/SilverWalk/artifacts/reports/mlp_positive_risk_regressor_condensed_nearest_neighbour_two_stage_history.csv --predictions-path /home/huichan/SPARV/SilverWalk/artifacts/predictions/mlp_positive_risk_regressor_condensed_nearest_neighbour_two_stage_test_predictions.csv --epochs 100 --

I0000 00:00:1780786876.672450   54941 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780786876.766027   54941 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780786878.024952   54941 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1780786879.520809   54941 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please mak

Epoch 1/100
62/62 - 1s - 16ms/step - loss: 0.3704 - mae: 0.7191 - rmse: 0.9639 - val_loss: 0.3387 - val_mae: 0.6661 - val_rmse: 0.9492 - learning_rate: 0.0010
Epoch 2/100
62/62 - 0s - 5ms/step - loss: 0.3019 - mae: 0.6327 - rmse: 0.8577 - val_loss: 0.2921 - val_mae: 0.6140 - val_rmse: 0.8630 - learning_rate: 0.0010
Epoch 3/100
62/62 - 0s - 5ms/step - loss: 0.2854 - mae: 0.6110 - rmse: 0.8310 - val_loss: 0.2718 - val_mae: 0.5890 - val_rmse: 0.8221 - learning_rate: 0.0010
Epoch 4/100
62/62 - 0s - 5ms/step - loss: 0.2777 - mae: 0.6003 - rmse: 0.8172 - val_loss: 0.2614 - val_mae: 0.5759 - val_rmse: 0.8010 - learning_rate: 0.0010
Epoch 5/100
62/62 - 0s - 5ms/step - loss: 0.2706 - mae: 0.5913 - rmse: 0.8049 - val_loss: 0.2544 - val_mae: 0.5657 - val_rmse: 0.7877 - learning_rate: 0.0010
Epoch 6/100
62/62 - 0s - 5ms/step - loss: 0.2642 - mae: 0.5820 - rmse: 0.7943 - val_loss: 0.2481 - val_mae: 0.5579 - val_rmse: 0.7737 - learning_rate: 0.0010
Epoch 7/100
62/62 - 0s - 5ms/step - loss: 0.2587 - 

## 결과 지표 표 위치

실험을 실행한 뒤 아래 지표만 `docs/기법별_성능평가.md`의 평가표에 작성한다.

| 모델 | PR-AUC | Recall@Top10% | Precision@Top10% | MAE | RMSE |
| --- | ---: | ---: | ---: | ---: | ---: |
| 실험명 | 실행 후 입력 | 실행 후 입력 | 실행 후 입력 | 실행 후 입력 | 실행 후 입력 |


In [9]:
def top_percent_binary_metrics(predictions: pd.DataFrame, score_column: str, target_column: str, percent: float = 0.10) -> dict[str, float]:
    """예측 점수 상위 percent 구간의 precision/recall을 계산한다."""
    if predictions.empty:
        return {'precision_top10pct': math.nan, 'recall_top10pct': math.nan}

    k = max(1, math.ceil(len(predictions) * percent))
    ranked = predictions.sort_values(score_column, ascending=False).head(k)
    actual_positive = predictions[target_column].astype(int)
    top_positive = ranked[target_column].astype(int)

    precision = float(top_positive.sum() / k)
    recall = float(top_positive.sum() / actual_positive.sum()) if actual_positive.sum() else math.nan
    return {'precision_top10pct': precision, 'recall_top10pct': recall}


def rmse(y_true: pd.Series, y_pred: pd.Series) -> float:
    """scikit-learn 버전 차이를 피하기 위해 RMSE를 직접 계산한다."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


if MODEL1_PREDICTIONS_PATH.exists() and MODEL2_PREDICTIONS_PATH.exists():
    model1_predictions = pd.read_csv(MODEL1_PREDICTIONS_PATH)
    model2_predictions = pd.read_csv(MODEL2_PREDICTIONS_PATH)

    model1_summary = top_percent_binary_metrics(
        model1_predictions,
        score_column='사고발생확률',
        target_column='사고발생',
        percent=0.10,
    )
    model1_summary['pr_auc'] = float(
        average_precision_score(model1_predictions['사고발생'], model1_predictions['사고발생확률'])
    )

    model2_summary = {
        'mae': float(mean_absolute_error(model2_predictions['위험도'], model2_predictions['pred_위험도'])),
        'rmse': rmse(model2_predictions['위험도'], model2_predictions['pred_위험도']),
    }

    summary = pd.DataFrame([
        {
            '모델': EXPERIMENT_NAME,
            'PR-AUC': model1_summary['pr_auc'],
            'Recall@Top10%': model1_summary['recall_top10pct'],
            'Precision@Top10%': model1_summary['precision_top10pct'],
            'MAE': model2_summary['mae'],
            'RMSE': model2_summary['rmse'],
        }
    ])

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(EXPERIMENT_RESULT_PATH, index=False)
    display(summary)
    print(f'실험 요약 저장: {EXPERIMENT_RESULT_PATH.relative_to(PROJECT_ROOT)}')
    print(f'문서 평가표 위치: {DOCS_TABLE_PATH.relative_to(PROJECT_ROOT)}')
else:
    print('아직 모델 예측 파일이 없습니다. RUN_SAMPLING=True, RUN_TRAINING=True로 학습을 마친 뒤 다시 실행하십시오.')
    print(f'필요 파일: {MODEL1_PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}')
    print(f'필요 파일: {MODEL2_PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}')


,모델,PR-AUC,Recall@Top10%,Precision@Top10%,MAE,RMSE
0,Undersampling CNN + 모델 2개,0.434699,0.261015,0.520562,2.026514,6.303183


실험 요약 저장: artifacts/experiments/condensed_nearest_neighbour_two_stage/condensed_nearest_neighbour_two_stage_summary.csv
문서 평가표 위치: docs/기법별_성능평가.md
